New Clustering with rthema groups, but CQs can be in several groups repeatedly so connections are not lost, with Fabians Noise Handeling and the Cluster Labeling

In [40]:
import os
import re
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
from sentence_transformers import SentenceTransformer
import umap.umap_ as umap
import hdbscan
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_distances
import nltk

In [41]:
#Load Stopwords (standard of given package + our own "legal" additions)
nltk.download("stopwords")
from nltk.corpus import stopwords
base_stopwords = stopwords.words('german')
own_additions = [
    "welche", "hat", "daher", "damit", "ob", "um", "wie", "wer", "was", "wann", "wo", "wurde",
    "liegt", "besteht", "enthält", "stellt", "erfüllt", "bezieht", "basiert", "bestimmen", "bewerten",
    "gilt", "spielt", "tritt", "verlangt", "aufweist", "führt", "genannt", "verweist",
    "rechtsgrundlage", "rechtsfolge", "recht", "rechtlich", "rechtliche", "rechtlicher", "rechtlichen",
    "gesetzlich", "gerichtlich", "rechtsprechung", "rechtsprechungen",
    "anspruch", "ansprüche", "anspruchs", "ansprüchen", "anspruchsgrundlage", "anspruchsgrundlagen",
    "forderung", "forderungen", "entscheidung", "feststellung", "anwendung", "voraussetzungen",
    "urteils", "urteil", "klage", "verfahren", "geltend", "geltendmachung",
    "bgb", "zpo", "hgb", "egbgb", "prodhaftg", "hoai", "vglo", "satz", "abs", "nr", "art", "nach", "gemäß",
    "auch", "nicht", "sowie", "jedoch", "nur", "noch", "bereits", "alle", "mehr", "weniger", "einschließlich",
    "beklagten", "kläger", "klägerin", "beklagte", "parteien", "person", "personen",
    "vertrag", "vertrages", "vertrags", "vereinbarung", "vereinbart", "vereinbarten", "regelung", "rahmen",
    "pflichten", "rechte", "bedingungen", "bestimmung", "bestimmungen",
    "müssen", "dürfen", "sollen", "können", "dürfte", "wird", "sind", "sein", "verpflichtet", "unwirksam", "wirksam",
    "rolle", "bedeutung", "wichtige", "wichtiger", "insbesondere", "einschlägig", "wesentliche", "wesentlichen",
    "warum", "auf", "hinblick", "betreffend", "weiteren", "anklageschrift", "abweichen", "geltung", "geltungsbereich"
]
stopwords_de = list(set(base_stopwords + own_additions))

#Synonym-Mapping to be extended if needed
synonym_map = {
    "vergütung": ["entlohnung", "honorar", "bezahlung"]
}

#Function to use Synonyms later
def apply_synonyms(texts, mapping):
    for i in range(len(texts)):
        for target, syns in mapping.items():
            for syn in syns:
                texts[i] = texts[i].replace(syn, target)
    return texts

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nicografvonwedel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [42]:
#Load all Competency Questions
input_path = Path("competency_questions_output/competency_questions_all_documents.txt")
with open(input_path, "r", encoding="utf-8") as f:
    full_text = f.read()


In [43]:
#Split the Document into Rechtsthema Groups
documents = re.split(r"\nDokument:\s", full_text)
rthema_to_cqs = defaultdict(list)

for doc in documents:
    if not doc.strip(): continue
    rthema_match = re.search(r"Rechtsthema:\s(.+)", doc)
    if not rthema_match: continue
    rthemen = [t.strip() for t in rthema_match.group(1).split(",")]
    questions = re.findall(r"\*\*Frage:\*\*\s*(.+)", doc)
    for thema in rthemen:
        rthema_to_cqs[thema].extend(questions)

In [44]:
#Load Embedding Model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

#Create Output Folder
os.makedirs("cluster_output_by_rthema", exist_ok=True)
summary = {}

In [45]:
#Clustering & Labeling per Rechtsthema
def process_thema(thema, questions):
    #Generate embeddings
    embeddings = model.encode(questions)
    embeddings = normalize(embeddings)

    #Dimensionality reduction
    reducer = umap.UMAP(n_neighbors=30, n_components=3, metric="cosine", random_state=42)
    reduced = reducer.fit_transform(embeddings)

    #Clustering with HDBSCAN
    clusterer = hdbscan.HDBSCAN(min_cluster_size=8, min_samples=5, metric="euclidean")
    labels = clusterer.fit_predict(reduced)

    clustered_data = defaultdict(list)
    noise_data = []

    #Create Count Variables for Noise Triage
    promoted_count = 0
    reviewed_count = 0

    #Assign questions to clusters or noise
    for idx, label in enumerate(labels):
        entry = {"cq": questions[idx], "embedding": embeddings[idx].tolist()}
        if label == -1:
            noise_data.append(entry)
        else:
            clustered_data[f"Cluster {label}"].append(entry)

    #Handle Noise Triage
    if noise_data:
        PROMOTE_THRESHOLD = 0.2
        REVIEW_THRESHOLD = 0.25

        noise_embeddings = np.array([x["embedding"] for x in noise_data])
        core_embeddings = []
        idx_to_cluster = {}
        idx = 0
        for cname, items in clustered_data.items():
            for cq in items:
                core_embeddings.append(cq["embedding"])
                idx_to_cluster[idx] = cname
                idx += 1
        core_embeddings = np.array(core_embeddings)

        dist_matrix = cosine_distances(noise_embeddings, core_embeddings)
        d_min = dist_matrix.min(axis=1)
        nearest_idx = dist_matrix.argmin(axis=1)

        for i, entry in enumerate(noise_data):
            dist = d_min[i]
            nearest_cluster = idx_to_cluster[nearest_idx[i]]
            tag = "promote" if dist <= PROMOTE_THRESHOLD else "review" if dist <= REVIEW_THRESHOLD else "leave"
            entry["from_noise"] = tag
            if tag == "promote":
                promoted_count += 1
                clustered_data[nearest_cluster].append(entry)
            elif tag == "review":
                reviewed_count += 1
                clustered_data[nearest_cluster].append(entry)
            else:
                clustered_data["Cluster -1"].append(entry)


    #TF-IDF Labeling of our Clusters
    vectorizer = TfidfVectorizer(
    stop_words=stopwords_de,
    token_pattern=r'(?u)\b[^\d\W]{2,}\b')
    all_texts = [entry["cq"] for cl in clustered_data.values() for entry in cl]
    all_texts = apply_synonyms(all_texts, synonym_map)
    vectorizer.fit(all_texts)
    feature_names = np.array(vectorizer.get_feature_names_out())

    labeled_clusters = {}
    cluster_keywords = {}
    for cluster_id, items in clustered_data.items():
        texts = [e["cq"] for e in items]
        texts = apply_synonyms(texts, synonym_map)
        tfidf = vectorizer.transform(texts).mean(axis=0).A1
        keywords = feature_names[tfidf.argsort()[-7:][::-1]]
        labeled_clusters[cluster_id] = {
            "size": len(texts),
            "top_keywords": keywords.tolist(),
            "questions": texts
        }
        cluster_keywords[cluster_id] = keywords.tolist()

    #Save Cluster Keywords for Dashboard later
    with open(f"dashboard_data/cluster_keywords_{thema.replace('/', '_')}.json", "w", encoding="utf-8") as f:
        json.dump(cluster_keywords, f, ensure_ascii=False, indent=2)

    return labeled_clusters, len([x for x in noise_data if x["from_noise"] == "leave"]), promoted_count, reviewed_count

In [46]:
#Main Loop per Rechtsthema
for thema, questions in rthema_to_cqs.items():
    if len(questions) < 10:
        continue
    print(f"🔍 {thema}: {len(questions)} Fragen")
    labeled_clusters, noise_final, promoted, reviewed = process_thema(thema, questions)

    # Save clustered CQs per Rechtsthema
    cluster_output_path = Path("cluster_output_by_rthema") / f"clusters_{thema.replace('/', '_')}.json"
    with open(cluster_output_path, "w", encoding="utf-8") as f_out:
        json.dump(labeled_clusters, f_out, indent=2, ensure_ascii=False)

        print(f"🔍 {thema}: {len(questions)} Fragen → Clusters: {len(labeled_clusters)} | Final Noise: {noise_final} | Promoted: {promoted} | Reviewed: {reviewed}")
        summary[thema] = {
            "clusters": len(labeled_clusters),
            "total_cqs": len(questions),    
            "noise_final": noise_final,
            "noise_promoted": promoted,
            "noise_reviewed": reviewed
        }


🔍 Schadensersatz: 15954 Fragen


/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


🔍 Schadensersatz: 15954 Fragen → Clusters: 334 | Final Noise: 530 | Promoted: 3919 | Reviewed: 894
🔍 SchuldrechtAT: 17044 Fragen


/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
huggingface/tokenizers: The current proce

🔍 SchuldrechtAT: 17044 Fragen → Clusters: 295 | Final Noise: 479 | Promoted: 4225 | Reviewed: 843
🔍 Miete Pacht: 8441 Fragen


...

/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


🔍 Erbschaft Schenkung: 794 Fragen → Clusters: 29 | Final Noise: 62 | Promoted: 79 | Reviewed: 78
🔍 Sonstiges Recht: 400 Fragen


/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


🔍 Sonstiges Recht: 400 Fragen → Clusters: 2 | Final Noise: 0 | Promoted: 0 | Reviewed: 0


/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/nicografvonwedel/Documents/GitHub/CapstoneDATEV25/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [47]:
#Save Clustered Data to File in cluster_output_by_rthema Folder
output_path = Path("cluster_output_by_rthema") / f"clusters_{thema.replace('/', '_')}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(labeled_clusters, f, indent=2, ensure_ascii=False)

#Save Summary
with open("cluster_output_by_rthema/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n✅ Clustering is done – now with respective Noise shown per cluster + Noise Handeling + Cluster Labeling.")


✅ Clustering is done – now with respective Noise shown per cluster + Noise Handeling + Cluster Labeling.
